In [1]:
!pip install -U deepeval
!pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.5/812.5 kB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.37.0
    Uninstalling opentelemetry-proto-1.37.0:
      Successfully uninstalled opentelemetry-proto-1.37.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%bash
sudo apt-get update && sudo apt-get install zstd -y
curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,302 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,623 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:13 http://security.ubuntu.com/ubuntu j

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:114

In [4]:

# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')



ln: failed to create symbolic link '/root/.ollama/models': No such file or directory
Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models
Ollama server started as background process (PID: 3364 )
Waited 5 seconds for the server to initialize.


In [5]:
# LLM
!ollama pull gpt-oss:20b

!ollama pull nomic-embed-text

In [6]:
import pandas as pd
import os

# Increase the timeout for DeepEval operations
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import OllamaModel

In [7]:
# DeepEval has a native Ollama integration (no OpenAI-compatible /v1 needed).
# Use the Ollama base URL WITHOUT /v1.
judge_llm = OllamaModel(
    model="gpt-oss:20b",
    base_url="http://localhost:11434",
    temperature=0,
    # Optional: pass Ollama options if you need a larger context window
    # (Ollama uses num_ctx rather than "max_tokens")
    generation_kwargs={"options": {"num_ctx": 128000}},
)

In [8]:
eval_dataset_df = pd.read_excel(r"/content/drive/MyDrive/RAG/results/eval_result_combined.xlsx", sheet_name="final")

# Llama
eval_dataset_l2_df = eval_dataset_df[
    ["user_query", "ref_answer", "l2_response"]
].rename(
    columns={
        "user_query": "question",
        "l2_response": "answer",
        "ref_answer": "ground_truth",
    }
)

for col in ["question", "answer", "ground_truth"]:
    eval_dataset_l2_df[col] = eval_dataset_l2_df[col].astype(str)

# Gemma
eval_dataset_g2_df = eval_dataset_df[
    ["user_query", "ref_answer", "g2_response"]
].rename(
    columns={
        "user_query": "question",
        "g2_response": "answer",
        "ref_answer": "ground_truth",
    }
)

for col in ["question", "answer", "ground_truth"]:
    eval_dataset_g2_df[col] = eval_dataset_g2_df[col].astype(str)

# Deepseek
eval_dataset_d2_df = eval_dataset_df[
    ["user_query", "ref_answer", "d2_response"]
].rename(
    columns={
        "user_query": "question",
        "d2_response": "answer",
        "ref_answer": "ground_truth",
    }
)

for col in ["question", "answer", "ground_truth"]:
    eval_dataset_d2_df[col] = eval_dataset_d2_df[col].astype(str)


In [9]:
correctness_metric = GEval(
    name="Answer Correctness",
    model=judge_llm,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    criteria=(
        "Score how correct the ACTUAL_OUTPUT is relative to the EXPECTED_OUTPUT.\n"
        "Consider factual correctness and completeness.\n"
        "If ACTUAL_OUTPUT contradicts EXPECTED_OUTPUT, score low.\n"
        "If ACTUAL_OUTPUT matches EXPECTED_OUTPUT in meaning (paraphrase is OK), score high.\n"
        "Ignore stylistic differences.\n"
    ),
    # Keep strict_mode=False to preserve 0–1 scoring behavior
    strict_mode=False,
    async_mode=False,   # safer in notebooks; set True if you want internal async execution
)


In [10]:
%%time
from tqdm.auto import tqdm

scores = []
reasons = []

for row in tqdm(eval_dataset_l2_df.itertuples(index=False), total=len(eval_dataset_l2_df)):
    tc = LLMTestCase(
        input=row.question,
        actual_output=row.answer,
        expected_output=row.ground_truth,
    )
    s = correctness_metric.measure(tc)
    scores.append(s)
    reasons.append(getattr(correctness_metric, "reason", None))

eval_dataset_l2_df["answer_correctness_score"] = scores
eval_dataset_l2_df["answer_correctness_reason"] = reasons  # optional but usually useful

display(eval_dataset_l2_df[["answer_correctness_score"]].head())


  0%|          | 0/197 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

,answer_correctness_score
0,1.0
1,0.2
2,0.0
3,1.0
4,1.0


CPU times: user 49.7 s, sys: 3.09 s, total: 52.8 s
Wall time: 27min 48s


In [11]:
eval_dataset_l2_df.to_excel(r"/content/drive/MyDrive/RAG/results/deepeval_l2_ac_20260118.xlsx", index=False)

In [12]:
%%time
from tqdm.auto import tqdm

scores = []
reasons = []

for row in tqdm(eval_dataset_g2_df.itertuples(index=False), total=len(eval_dataset_g2_df)):
    tc = LLMTestCase(
        input=row.question,
        actual_output=row.answer,
        expected_output=row.ground_truth,
    )
    s = correctness_metric.measure(tc)
    scores.append(s)
    reasons.append(getattr(correctness_metric, "reason", None))

eval_dataset_g2_df["answer_correctness_score"] = scores
eval_dataset_g2_df["answer_correctness_reason"] = reasons  # optional but usually useful

display(eval_dataset_g2_df[["answer_correctness_score"]].head())


  0%|          | 0/197 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

,answer_correctness_score
0,1.0
1,1.0
2,0.0
3,0.0
4,0.8


CPU times: user 53.2 s, sys: 3.32 s, total: 56.5 s
Wall time: 28min 33s


In [13]:
eval_dataset_g2_df.to_excel(r"/content/drive/MyDrive/RAG/results/deepeval_g2_ac_20260118.xlsx", index=False)

In [14]:
%%time
from tqdm.auto import tqdm

scores = []
reasons = []

for row in tqdm(eval_dataset_d2_df.itertuples(index=False), total=len(eval_dataset_d2_df)):
    tc = LLMTestCase(
        input=row.question,
        actual_output=row.answer,
        expected_output=row.ground_truth,
    )
    s = correctness_metric.measure(tc)
    scores.append(s)
    reasons.append(getattr(correctness_metric, "reason", None))

eval_dataset_d2_df["answer_correctness_score"] = scores
eval_dataset_d2_df["answer_correctness_reason"] = reasons  # optional but usually useful

display(eval_dataset_d2_df[["answer_correctness_score"]].head())


  0%|          | 0/197 [00:00<?, ?it/s]

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

,answer_correctness_score
0,1.0
1,1.0
2,0.0
3,1.0
4,1.0


CPU times: user 54.5 s, sys: 3.53 s, total: 58 s
Wall time: 28min 30s


In [15]:
eval_dataset_d2_df.to_excel(r"/content/drive/MyDrive/RAG/results/deepeval_d2_ac_20260118.xlsx", index=False)
